In [ ]:
!git clone https://github.com/sara-graca/acoustic-vs-neural-representations.git
%cd acoustic-vs-neural-representations
!pip install tgt chardet

In [ ]:
import shutil
import os

os.makedirs("data/parsed", exist_ok=True)
os.makedirs("data/features", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)

shutil.copy(
    "/kaggle/input/datasets/sara2graca/statistics-parsed-data/phonemes.csv",
    "data/parsed/phonemes.csv"
)

shutil.copytree(
    "/kaggle/input/datasets/sara2graca/statistics-all-data/ru-fr_interference",
    "data/raw/ru-fr_interference"
)

print("Data ready")

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import torch
from transformers import WhisperProcessor, WhisperModel
from tqdm.notebook import tqdm
import librosa

MODEL_NAME = "openai/whisper-medium"
LAYER      = 4
INPUT      = "data/parsed/phonemes.csv"
OUTPUT     = "data/features/features_whisper_layer4.npz"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device}")

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model     = WhisperModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
model     = model.to(device)
model.eval()

df = pd.read_csv(INPUT)

# fix wav paths to point to kaggle data
df["wav_path"] = df["wav_path"].apply(
    lambda p: os.path.join(
        "/kaggle/input/datasets/sara2graca/statistics-all-data",
        p.replace("data\\raw\\", "").replace("data/raw/", "").replace("\\", "/")
    )
)

embeddings = {}
current_wav, current_audio, current_sr = None, None, None

for i, row in tqdm(df.iterrows(), total=len(df)):
    if row["wav_path"] != current_wav:
        current_wav = row["wav_path"]
        try:
            current_audio, current_sr = librosa.load(current_wav, sr=16000)
        except Exception:
            current_audio, current_sr = None, None
    
        if current_audio is None:
            embeddings[i] = np.zeros(model.config.d_model)
            continue

    onset_sample  = int(row["onset"] * current_sr)
    offset_sample = int(row["offset"] * current_sr)
    segment = current_audio[onset_sample:offset_sample]

    if len(segment) < 10:
        embeddings[i] = np.zeros(model.config.d_model)
        continue

    inputs = processor(segment, sampling_rate=current_sr, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.encoder(inputs.input_features, output_hidden_states=True)

    hidden   = outputs.hidden_states[LAYER]
    seq_len  = hidden.shape[1]

    total_frames = inputs.input_features.shape[-1] // 2
    start_frame  = min(int(row["onset"] * total_frames / 30.0), seq_len - 1)
    dur_frames   = max(1, int((row["offset"] - row["onset"]) * total_frames / 30.0))
    end_frame    = min(start_frame + dur_frames, seq_len)

    embedding     = hidden[0, start_frame:end_frame, :].mean(dim=0).cpu().numpy()
    embeddings[i] = embedding

os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)
np.savez(OUTPUT, **{str(k): v for k, v in embeddings.items()})
print(f"Done: {len(embeddings)} embeddings → {OUTPUT}")

shutil.copy("data/features/features_whisper_layer4.npz", "/kaggle/working/features_whisper_layer4.npz")